<a href="https://colab.research.google.com/github/muhnehh/flyrank-ml-internship-starter/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 - Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhnehh/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook turns the CTR opportunity lane into an explicit ML and ranking task. All rates use the dataset's documented percentage-point scale.

**Repo references used:** `docs/data-dictionary.md`, `docs/ml-intern-dataset-and-lane-guide.md`, `skills/framing-ml-problems/SKILL.md`, and `skills/flyrank/flyrank-data/SKILL.md`.

## 1. My lane as an ML task (type)

**Task type:** **Regression feeding a ranking system.**

The regression component estimates the CTR that a page would normally achieve given search position and safe contextual features. The business output is then a ranked queue, not a raw prediction table.

For each eligible page:

\[
	ext{CTR gap (percentage points)} = \widehat{	ext{expected CTR}} - 	ext{observed CTR}
\]

\[
	ext{directional click opportunity} =
\max(	ext{CTR gap}, 0) / 100 	imes 	ext{impressions}
\]

Pages with larger positive gaps, meaningful volume, and adequate measurement coverage receive higher review priority. This is expected-performance estimation and prioritization; it is not a causal traffic forecast.

In [ ]:
from pathlib import Path
import pandas as pd

# Works in Colab, a cloned repo, and the downloadable notebook package.
DATA_CANDIDATES = [
    Path('/content/content_refresh_anonymized.csv'),
    Path('/mnt/data/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        df = pd.read_csv(candidate)
        break
else:
    DATA_URL = (
        'https://raw.githubusercontent.com/muhnehh/'
        'flyrank-ml-internship-starter/main/data/raw/'
        'content_refresh_anonymized.csv'
    )
    df = pd.read_csv(DATA_URL)

print(f'Dataset loaded: {df.shape[0]:,} pages x {df.shape[1]} columns')
print(f'Clients: {df["client_id"].nunique():,}')
print(f'Unique content IDs: {df["content_id"].nunique():,}')


print('\nCTR summary (percentage points; 0.76 means 0.76%):')
print(df['ctr'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).round(3))
print(f"\nRows with zero observed CTR: {(df['ctr'] == 0).sum():,} / {len(df):,}")
print(f"Rows with avg_position == 0 (no position data): {(df['avg_position'] == 0).sum():,}")

Dataset loaded: 30,000 pages x 44 columns
Clients: 32
Unique content IDs: 30,000

CTR summary (percentage points; 0.76 means 0.76%):
count    30000.000
mean         0.511
std          3.279
min          0.000
25%          0.000
50%          0.070
75%          0.290
90%          0.650
95%          1.090
99%          8.330
max        100.000
Name: ctr, dtype: float64

Rows with zero observed CTR: 13,212 / 30,000
Rows with avg_position == 0 (no position data): 1,205


## 2. Target or proxy

- **Regression target:** `ctr`, stored in **percentage points**. For example, `ctr = 0.76` means **0.76%**.
- **Observed formula:** `100 * clicks_90d / impressions_90d`, rounded to two decimal places.
- **Label origin:** This is an observed 90-day outcome, not a manually assigned quality label.
- **Prediction scope:** In the starter project, the model estimates expected cross-sectional CTR from comparable pages. It does not yet predict a future month.
- **Excluded answer-bearing inputs:** `ctr`, `clicks_90d`, and any feature directly calculated from clicks must not be used to predict CTR.
- **Secondary guardrails:** `engagement_rate` and `scroll_rate` may help distinguish a search-snippet review from a post-click content review, but they do not verify whether the CTR estimate is correct.
- **Identifiers:** `content_id` and `client_id` are used for joins, grouping, and validation splits only - never as model features.

In [ ]:
import numpy as np

# Verify the documented percentage-point formula.
computed_ctr = np.where(
    df['impressions_90d'] > 0,
    100 * df['clicks_90d'] / df['impressions_90d'],
    0,
)
absolute_difference = np.abs(df['ctr'] - computed_ctr)

print(f"Maximum difference before allowing for stored rounding: {absolute_difference.max():.6f} pp")
print(f"Rows differing by more than 0.011 percentage points: {(absolute_difference > 0.011).sum():,}")
print(f"Missing target values: {df['ctr'].isna().sum():,}")
print("Interpretation: differences up to about 0.01 pp are expected because ctr is rounded.")

Maximum difference before allowing for stored rounding: 0.005000 pp
Rows differing by more than 0.011 percentage points: 0
Missing target values: 0
Interpretation: differences up to about 0.01 pp are expected because ctr is rounded.


## 3. Success metric

### Primary model metric

**Client-holdout MAE in CTR percentage points.** Entire clients are held out so pages from the same client do not appear in both training and evaluation data.

### Baseline

A transparent baseline predicts the **training-set median CTR for the page's position tier**, with a training global median as fallback. A global median alone is too weak because CTR depends strongly on position.

### Provisional success rule

The first learned model should:

1. reduce client-holdout MAE by at least **10% relative to the position-tier baseline**;
2. also improve or remain stable on impression-weighted MAE;
3. produce a sensible top review queue without depending on leakage features;
4. remain directionally stable across clients and reasonable volume floors.

The 10% threshold is a provisional engineering target, not a fact derived from the data. It may be revised after repeated validation.

### Final decision metric

The final system is a ranking. The long-term metric should therefore include top-K review quality or later measured outcomes after editorial review. The current historical starter dataset alone does not contain those intervention outcomes.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Eligibility keeps extreme low-volume noise and missing position data out of the first baseline.
eligible = df[
    (df['impressions_90d'] >= 100)
    & (df['avg_position'] > 0)
].copy()

# Deterministic client-holdout split.
unique_clients = eligible['client_id'].drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

train = eligible[~eligible['client_id'].isin(test_clients)].copy()
test = eligible[eligible['client_id'].isin(test_clients)].copy()

# Fit the baseline ONLY on the training clients.
position_medians = train.groupby('position_tier')['ctr'].median()
global_fallback = train['ctr'].median()
baseline_prediction = test['position_tier'].map(position_medians).fillna(global_fallback)

baseline_mae = mean_absolute_error(test['ctr'], baseline_prediction)
baseline_weighted_mae = np.average(
    np.abs(test['ctr'] - baseline_prediction),
    weights=test['impressions_90d'],
)

print(f'Eligible pages: {len(eligible):,}')
print(f'Train pages: {len(train):,} across {train.client_id.nunique()} clients')
print(f'Test pages: {len(test):,} across {test.client_id.nunique()} held-out clients')
print(f'Position-tier baseline MAE: {baseline_mae:.4f} percentage points')
print(f'Impression-weighted MAE: {baseline_weighted_mae:.4f} percentage points')
print('\nTraining-set median CTR by position tier:')
print(position_medians.sort_values(ascending=False).to_string())

Eligible pages: 22,006
Train pages: 20,985 across 24 clients
Test pages: 1,021 across 6 held-out clients
Position-tier baseline MAE: 0.2564 percentage points
Impression-weighted MAE: 0.2296 percentage points

Training-set median CTR by position tier:
position_tier
page_1      0.22
top_3       0.18
striking    0.15
page_3_5    0.06
deep        0.00


## 4. The unit of analysis, as a real dataframe

**Unit of analysis:** one unique pseudonymized web page (`content_id`).

Each starter row aggregates metadata and search/analytics measurements over a trailing 90-day window. The starter CSV is therefore page-level and cross-sectional. The later warehouse release has a different grain - one date x client x content item - and must be aggregated into explicit feature and target windows before modeling.

In [ ]:
# Present a real page-level slice while keeping all identifiers pseudonymized.
columns_to_show = [
    'content_id', 'client_id', 'content_type', 'main_intent',
    'avg_position', 'impressions_90d', 'clicks_90d', 'ctr',
    'engagement_rate', 'days_since_last_update'
]

print(f'Full dataset shape: {df.shape}')
print(f'Is content_id unique per row? {df.content_id.nunique() == len(df)}')
print(f'Number of pseudonymized clients: {df.client_id.nunique()}')
df[columns_to_show].head(5)

Full dataset shape: (30000, 44)
Is content_id unique per row? True
Number of pseudonymized clients: 32


,content_id,client_id,content_type,main_intent,avg_position,impressions_90d,clicks_90d,ctr,engagement_rate,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.6,3803,29,0.76,5.88,20
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,20.3,15320,7,0.05,0.00,25
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,36.5,12581,11,0.09,0.00,20
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,6.2,11751,58,0.49,1.28,22
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,44.0,19140,24,0.13,0.00,14


## 5. Why ML may beat a fixed rule here

A fixed rule such as `avg_position <= 10 AND ctr < 2.0` treats unlike pages as if they were comparable. It also uses a hard boundary even though CTR and measurement reliability change gradually.

A stronger workflow is:

1. apply eligibility and volume rules;
2. build a transparent position-adjusted baseline;
3. test whether an interpretable model improves held-out-client error;
4. convert positive residuals into a ranked queue with reason codes;
5. retain the baseline if the learned model does not improve honestly.

Potential safe model inputs include `avg_position`, `position_tier`, `main_intent`, `content_type`, `search_volume`, `competition`, `content_age_days`, `days_since_last_update`, `freshness_tier`, and content-length fields with explicit missingness indicators.

Do **not** use `ctr`, `clicks_90d`, `content_id`, or `client_id` as predictive features. Use impressions mainly for eligibility, weighting, confidence, and final opportunity sizing rather than allowing raw volume to dominate the expected-CTR model.

In [ ]:
# Show why one universal CTR rule is inadequate.
# We use a volume floor and report only groups with enough pages to reduce obvious noise.
comparison = (
    df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)]
    .groupby(['position_tier', 'main_intent'], dropna=False)['ctr']
    .agg(pages='count', median_ctr='median', mean_ctr='mean', std_ctr='std')
    .reset_index()
)
comparison = comparison[comparison['pages'] >= 50].sort_values(
    ['position_tier', 'median_ctr'], ascending=[True, False]
)

print('CTR varies across position tiers and intent groups (percentage points):')
comparison.head(20).round(3)

CTR varies across position tiers and intent groups (percentage points):


,position_tier,main_intent,pages,median_ctr,mean_ctr,std_ctr
0,deep,commercial,150,0.00,0.047,0.138
1,deep,informational,542,0.00,0.057,0.179
3,deep,transactional,174,0.00,0.059,0.172
9,page_1,NaN,256,0.26,0.896,1.765
8,page_1,transactional,2006,0.25,0.359,0.394
5,page_1,commercial,1427,0.22,0.323,0.376
6,page_1,informational,4936,0.22,0.334,0.400
13,page_3_5,transactional,1113,0.07,0.159,0.257
10,page_3_5,commercial,1040,0.06,0.146,0.244
11,page_3_5,informational,3813,0.06,0.136,0.214


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled - markdown thinking and executable evidence.
- [x] CTR is interpreted in percentage points (`0.76` means `0.76%`).
- [x] The formula check includes `x 100`.
- [x] The baseline is position-adjusted and evaluated on held-out clients.
- [x] Leakage features and identifiers are explicitly excluded.
- [x] Claims remain observed, directional, and decision-support only.
- [ ] Save this file to `work/notebooks/w02_ml_task_framing.ipynb` in your repo and submit the repo URL on the assignment card.